In [21]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName('pyspark-pilot')
    .master('spark://spark-master:7077')
    .config('spark.executor.instances', 4)
    .config('spark.executor.cores', 2)
    .config('spark.executor.memory', '2048M')
    .getOrCreate()
)

spark

In [22]:
spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", False)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [30]:
spark.sparkContext.defaultParallelism

8

In [24]:
schema = "first_name string, last_name string, job_title string, dob string, email string, phone string, salary double, department_id int"

emp_df = spark.read.schema(schema).csv('/home/spark/data/employee/4', header=True)

In [25]:
emp_df.printSchema()

root
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- job_title: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- department_id: integer (nullable = true)



In [6]:
emp_df.show()

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            8|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            7|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|           10|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            1|
|  Michelle|   Elliott|      Air cabin crew|1975-03-31|tiffanyjohnston@e...|       (705)900-5337|604738.0|            8|
|    Ashley|   Montoya|        C

In [26]:
from pyspark.sql.functions import avg, col

emp_avg = emp_df.groupBy('department_id').agg(avg('salary').alias('avg_salary'))

In [8]:
emp_avg.show()

+-------------+------------------+
|department_id|        avg_salary|
+-------------+------------------+
|            1|504876.96401242825|
|            6|504428.12590014644|
|            3| 504697.6808514883|
|            5| 504167.9429997006|
|            9| 504945.3055672206|
|            4| 505419.4963977089|
|            8| 505299.1226286386|
|            7|504514.38453985273|
|           10| 502682.2575766687|
|            2| 503563.2174529479|
+-------------+------------------+



In [33]:
emp_avg.explain()

== Physical Plan ==
*(2) HashAggregate(keys=[department_id#103], functions=[avg(salary#102)])
+- Exchange hashpartitioning(department_id#103, 200), ENSURE_REQUIREMENTS, [plan_id=116]
   +- *(1) HashAggregate(keys=[department_id#103], functions=[partial_avg(salary#102)])
      +- FileScan csv [salary#102,department_id#103] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/spark/data/employee/4], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<salary:double,department_id:int>




In [31]:
spark.conf.set('spark.sql.shuffle.partitions', 8)
spark.conf.get('spark.sql.shuffle.partitions')

'8'

In [32]:
emp_avg.write.mode('overwrite').format('noop').save()

In [20]:
spark.stop()